In [ ]:
!ls /kaggle/input/

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path 
import warnings
import numpy as np 

warnings.filterwarnings('ignore')

# --- STEP 1: DEEP SEARCH FOR AUDIO ---
audio_files_info = []
search_root = "/kaggle/input/"

print("🔍 Searching deep for audio files...")

for root, dirs, files in os.walk(search_root):
    # Check if there are any audio files in this specific folder
    audio_in_folder = [f for f in files if f.lower().endswith(('.wav', '.mp3', '.m4a'))]
    
    if audio_in_folder:
        category = os.path.basename(root)
        # Skip generic root folders, we want the category folders
        if category == "input" or category == "baby-cry-dataset":
            continue
            
        for file in audio_in_folder:
            audio_files_info.append({
                'category': category,
                'filename': file,
                'file_path': os.path.join(root, file), 
                'extension': os.path.splitext(file)[1].lower() 
            })

df_audio = pd.DataFrame(audio_files_info)

# --- STEP 2: VERIFY AND PLOT ---
if not df_audio.empty:
    print(f"✅ SUCCESS! Found {len(df_audio)} audio files.")
    print(f"Detected Categories: {df_audio['category'].unique().tolist()}")
    
    # Plot Distribution
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df_audio, x='category', palette='viridis')
    plt.title('Audio Samples per Category')
    plt.xticks(rotation=45)
    plt.show()

    # --- STEP 3: VISUALIZE ONE SAMPLE ---
    sample = df_audio.sample(1).iloc[0]
    print(f"Analyzing Sample: {sample['filename']} ({sample['category']})")
    
    y, sr = librosa.load(sample['file_path'], sr=None, duration=5.0)
    
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title('Waveform')

    plt.subplot(1, 2, 2)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel Spectrogram')
    plt.tight_layout()
    plt.show()

    display(ipd.Audio(sample['file_path']))

else:
    print("❌ STILL NO DATA. Please check the 'Input' section on your right sidebar.")
    print("Is the dataset 'Added'? Current items in /kaggle/input/:", os.listdir('/kaggle/input/'))

In [ ]:
# 3- Convert to .WAV Files
import os
from pathlib import Path
try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    print("pydub library is not installed. Please run: !pip install pydub")
    PYDUB_AVAILABLE = False
import shutil

INPUT_AUDIO_DIRECTORY = Path("/kaggle/input/baby-cry-dataset")
OUTPUT_WAV_DIRECTORY = Path("/kaggle/working/Processed_WAV_Audio")

TARGET_FORMAT = "wav" 

def convert_audio_to_wav(source_dir: Path, target_dir: Path):
    if not PYDUB_AVAILABLE:
        print("pydub is not available. Conversion cannot proceed.")
        return 0, 0, 0, 0 

    if not source_dir.exists():
        print(f"Error: Input directory '{source_dir}' does not exist.")
        return 0, 0, 0, 0

    target_dir.mkdir(parents=True, exist_ok=True) 

    all_files = [p for p in source_dir.rglob('*') if p.is_file()]
    total_files = len(all_files)
    converted_count = 0
    copied_count = 0
    failed_count = 0

    print(f"Processing {total_files} files from '{source_dir}'...")

    for original_file_path in all_files:
        relative_path = original_file_path.relative_to(source_dir)
        output_subdir = target_dir / relative_path.parent
        output_subdir.mkdir(parents=True, exist_ok=True)

        original_extension = original_file_path.suffix.lower().lstrip('.')

        if original_extension == TARGET_FORMAT:
            output_file_path = output_subdir / original_file_path.name
            try:
                if not output_file_path.exists() or os.path.getsize(original_file_path) != os.path.getsize(output_file_path):
                    shutil.copy2(original_file_path, output_file_path)
                copied_count += 1
            except Exception as e:
                failed_count += 1
            continue

        new_wav_filename = original_file_path.stem + "." + TARGET_FORMAT
        output_file_path = output_subdir / new_wav_filename

        try:
            audio = AudioSegment.from_file(original_file_path, format=original_extension if original_extension else None)
            audio.export(output_file_path, format=TARGET_FORMAT)
            converted_count += 1
        except Exception as e:
            failed_count += 1

    print("Processing complete.")
    return total_files, converted_count, copied_count, failed_count

if __name__ == "__main__":
    if PYDUB_AVAILABLE:
        print(f"Input audio directory: {INPUT_AUDIO_DIRECTORY}")
        print(f"Output WAV directory: {OUTPUT_WAV_DIRECTORY}\n")

        total, converted, copied, failed = convert_audio_to_wav(INPUT_AUDIO_DIRECTORY, OUTPUT_WAV_DIRECTORY)

        print("\n--- Conversion Summary ---")
        print(f"Total files found in input: {total}")
        print(f"Successfully converted to WAV: {converted}")
        print(f"Already WAV and copied: {copied}")
        print(f"Failed to process: {failed}")
        if failed > 0:
            print("Review logs or uncomment debug print statements in the function for details on failures.")
        print(f"All processed WAV files are located in: {OUTPUT_WAV_DIRECTORY}")
    else:
        print("Script cannot run as pydub is not installed or importable.")

In [ ]:
# 4- Check file channels and convert them to mono channels
import os
from pathlib import Path
import librosa
import soundfile as sf 
import numpy as np

AUDIO_DIRECTORY_TO_PROCESS = Path("/kaggle/working/Processed_WAV_Audio")

OPERATION_MODE = 2  

MONO_OUTPUT_DIRECTORY_BASE = Path("/kaggle/working/MONO Files")
MONO_OUTPUT_FOLDER_NAME = "All_Mono_Audio"
FINAL_MONO_OUTPUT_PATH = MONO_OUTPUT_DIRECTORY_BASE / MONO_OUTPUT_FOLDER_NAME

def convert_to_mono_if_needed(audio_dir: Path, operation_mode: int, mono_output_dir: Path = None):
    if not audio_dir.exists():
        print(f"Error: Input directory '{audio_dir}' does not exist.")
        return 0, 0, 0, 0 

    if operation_mode == 2 and mono_output_dir is None:
        print("Error: For OPERATION_MODE 2, a 'mono_output_dir' must be specified.")
        return 0, 0, 0, 0
    if operation_mode == 2:
        mono_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output for all-mono files will be in: {mono_output_dir}")

    all_wav_files = [p for p in audio_dir.rglob('*.wav') if p.is_file()]
    total_files = len(all_wav_files)

    if total_files == 0:
        print(f"No .wav files found in '{audio_dir}'.")
        return 0, 0, 0, 0

    print(f"Processing {total_files} WAV files in '{audio_dir}' to ensure they are mono...")

    converted_to_mono_count = 0
    already_mono_count = 0
    failed_processing_count = 0

    processed_file_idx = 0
    for wav_file_path in all_wav_files:
        processed_file_idx += 1
        if processed_file_idx % 100 == 0 or processed_file_idx == total_files:
            print(f"  Processing file {processed_file_idx}/{total_files}...")
        try:
            y, sr = librosa.load(wav_file_path, sr=None, mono=False)

            is_mono = True
            if y.ndim > 1 and y.shape[0] > 1: 
                is_mono = False
                y_mono = librosa.to_mono(y)
                
            if operation_mode == 1: 
                output_path_for_this_file = wav_file_path
            else: 
                relative_path = wav_file_path.relative_to(audio_dir)
                output_path_for_this_file = mono_output_dir / relative_path
                output_path_for_this_file.parent.mkdir(parents=True, exist_ok=True)

            if not is_mono:
                sf.write(output_path_for_this_file, y_mono, sr)
                converted_to_mono_count += 1
            else:
                if operation_mode == 2 and wav_file_path != output_path_for_this_file:
                    shutil.copy2(wav_file_path, output_path_for_this_file)
                already_mono_count += 1

        except Exception as e:
            print(f"  ! Error processing/converting file {wav_file_path.name}: {e}")
            failed_processing_count += 1

    print("Mono conversion/check process complete.")
    return total_files, converted_to_mono_count, already_mono_count, failed_processing_count

if __name__ == "__main__":
    print(f"Input Audio Directory: {AUDIO_DIRECTORY_TO_PROCESS}")
    print(f"Operation Mode: {OPERATION_MODE} (1=In-place, 2=New Output Dir)")
    if OPERATION_MODE == 2:
        print(f"Target Mono Output Directory: {FINAL_MONO_OUTPUT_PATH}\n")
    else:
        print("Files will be modified in-place.\n")


    total, converted, already_mono, failed = convert_to_mono_if_needed(
        AUDIO_DIRECTORY_TO_PROCESS,
        OPERATION_MODE,
        FINAL_MONO_OUTPUT_PATH if OPERATION_MODE == 2 else None
    )

    print("\n--- Mono Conversion Summary ---")
    print(f"Total WAV files processed: {total}")
    print(f"Files converted to mono: {converted}")
    print(f"Files already mono: {already_mono}")
    print(f"Files failed to process: {failed}")

    if failed > 0:
        print("Review logs or uncomment debug print statements for details on failures.")

    if OPERATION_MODE == 1:
        print(f"Non-mono files in '{AUDIO_DIRECTORY_TO_PROCESS}' have been overwritten with mono versions.")
    elif OPERATION_MODE == 2 and total > 0 : 
        print(f"All processed files (ensured to be mono) are located in: {FINAL_MONO_OUTPUT_PATH}")

In [ ]:
# 5- Sample rate standardization for 16000 Hz
import os
from pathlib import Path
import librosa
import soundfile as sf 

MONO_AUDIO_DIRECTORY = Path("/kaggle/working/MONO Files/All_Mono_Audio/") 

TARGET_SR = 16000  

OPERATION_MODE = 2  

RESAMPLED_OUTPUT_DIRECTORY_BASE = Path("/kaggle/working/Processed_WAV_Audio")
RESAMPLED_OUTPUT_FOLDER_NAME = f"Resampled_SR{TARGET_SR//1000}kHz_Mono_Audio" 
FINAL_RESAMPLED_OUTPUT_PATH = RESAMPLED_OUTPUT_DIRECTORY_BASE / RESAMPLED_OUTPUT_FOLDER_NAME

def resample_audio_files(audio_dir: Path, target_sr: int, operation_mode: int, resampled_output_dir: Path = None):
    if not audio_dir.exists():
        print(f"Error: Input directory '{audio_dir}' does not exist.")
        return 0, 0, 0, 0 

    if operation_mode == 2 and resampled_output_dir is None:
        print("Error: For OPERATION_MODE 2, a 'resampled_output_dir' must be specified.")
        return 0, 0, 0, 0
    if operation_mode == 2:
        resampled_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output for resampled files will be in: {resampled_output_dir}")

    all_wav_files = [p for p in audio_dir.rglob('*.wav') if p.is_file()]
    total_files = len(all_wav_files)

    if total_files == 0:
        print(f"No .wav files found in '{audio_dir}'.")
        return 0, 0, 0, 0

    print(f"Processing {total_files} WAV files in '{audio_dir}' to resample to {target_sr} Hz...")

    resampled_count = 0
    already_correct_sr_count = 0
    failed_processing_count = 0

    processed_file_idx = 0
    for wav_file_path in all_wav_files:
        processed_file_idx += 1
        if processed_file_idx % 100 == 0 or processed_file_idx == total_files:
            print(f"  Processing file {processed_file_idx}/{total_files}...")
        try:
            y, sr_orig = librosa.load(wav_file_path, sr=None, mono=True)

            if operation_mode == 1: 
                output_path_for_this_file = wav_file_path
            else: 
                relative_path = wav_file_path.relative_to(audio_dir)
                output_path_for_this_file = resampled_output_dir / relative_path
                output_path_for_this_file.parent.mkdir(parents=True, exist_ok=True)

            if sr_orig != target_sr:
                y_resampled = librosa.resample(y, orig_sr=sr_orig, target_sr=target_sr)
                sf.write(output_path_for_this_file, y_resampled, target_sr)
                resampled_count += 1
            else:
                if operation_mode == 2 and wav_file_path != output_path_for_this_file:
                    shutil.copy2(wav_file_path, output_path_for_this_file)
                already_correct_sr_count += 1

        except Exception as e:
            print(f"  ! Error processing/resampling file {wav_file_path.name}: {e}")
            failed_processing_count += 1

    print("Resampling process complete.")
    return total_files, resampled_count, already_correct_sr_count, failed_processing_count

if __name__ == "__main__":
    print(f"Input Mono Audio Directory: {MONO_AUDIO_DIRECTORY}")
    print(f"Target Sample Rate: {TARGET_SR} Hz")
    print(f"Operation Mode: {OPERATION_MODE} (1=In-place, 2=New Output Dir)")
    if OPERATION_MODE == 2:
        print(f"Target Resampled Output Directory: {FINAL_RESAMPLED_OUTPUT_PATH}\n")
    else:
        print("Files will be modified in-place if their sample rate differs.\n")

    total, resampled, already_sr, failed = resample_audio_files(
        MONO_AUDIO_DIRECTORY,
        TARGET_SR,
        OPERATION_MODE,
        FINAL_RESAMPLED_OUTPUT_PATH if OPERATION_MODE == 2 else None
    )

    print("\n--- Resampling Summary ---")
    print(f"Total WAV files processed: {total}")
    print(f"Files resampled to {TARGET_SR} Hz: {resampled}")
    print(f"Files already at {TARGET_SR} Hz (and copied if mode 2): {already_sr}")
    print(f"Files failed to process: {failed}")

    if failed > 0:
        print("Review logs or uncomment debug print statements for details on failures.")

    if OPERATION_MODE == 1:
        print(f"Files in '{MONO_AUDIO_DIRECTORY}' with differing sample rates have been overwritten.")
    elif OPERATION_MODE == 2 and total > 0:
        print(f"All processed files (ensured to be at {TARGET_SR} Hz and mono) are located in: {FINAL_RESAMPLED_OUTPUT_PATH}")

In [ ]:
# 6- Unify duration to 7 seconds
import os
from pathlib import Path
import librosa
import soundfile as sf
import numpy as np

UNIFORM_SR_AUDIO_DIRECTORY = Path("/kaggle/working/Processed_WAV_Audio")

TARGET_DURATION_SECONDS = 7.0

EXPECTED_SAMPLE_RATE = 16000 

OPERATION_MODE = 2 

FIXED_DURATION_OUTPUT_DIRECTORY_BASE = Path("/kaggle/working/Unify duration to 7 seconds")
FIXED_DURATION_OUTPUT_FOLDER_NAME = f"FixedDuration_{int(TARGET_DURATION_SECONDS)}s_SR{EXPECTED_SAMPLE_RATE//1000}kHz_Mono_Audio"
FINAL_FIXED_DURATION_OUTPUT_PATH = FIXED_DURATION_OUTPUT_DIRECTORY_BASE / FIXED_DURATION_OUTPUT_FOLDER_NAME

def adjust_audio_duration(audio_dir: Path,
                           target_duration_s: float,
                           sample_rate: int,
                           operation_mode: int,
                           output_dir: Path = None):

    if not audio_dir.exists():
        print(f"Error: Input directory '{audio_dir}' does not exist.")
        return 0, 0, 0, 0, 0 

    if operation_mode == 2 and output_dir is None:
        print("Error: For OPERATION_MODE 2, an 'output_dir' must be specified.")
        return 0, 0, 0, 0, 0
    if operation_mode == 2:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output for fixed-duration files will be in: {output_dir}")

    all_wav_files = [p for p in audio_dir.rglob('*.wav') if p.is_file()]
    total_files = len(all_wav_files)

    if total_files == 0:
        print(f"No .wav files found in '{audio_dir}'.")
        return 0, 0, 0, 0, 0

    print(f"Processing {total_files} WAV files in '{audio_dir}' to adjust duration to {target_duration_s}s...")

    target_length_samples = int(target_duration_s * sample_rate)

    padded_count = 0
    truncated_count = 0
    already_correct_duration_count = 0
    failed_processing_count = 0

    processed_file_idx = 0
    for wav_file_path in all_wav_files:
        processed_file_idx += 1
        if processed_file_idx % 100 == 0 or processed_file_idx == total_files:
            print(f"  Processing file {processed_file_idx}/{total_files}...")
        try:
            y, sr = librosa.load(wav_file_path, sr=sample_rate, mono=True)

            if sr != sample_rate: 
                print(f"  Warning: File {wav_file_path.name} has SR {sr} != expected {sample_rate}. Resampling again.")
                y = librosa.resample(y, orig_sr=sr, target_sr=sample_rate)


            current_length_samples = len(y)
            output_audio = y 

            if current_length_samples < target_length_samples:
                padding_needed = target_length_samples - current_length_samples
                output_audio = np.pad(y, (0, padding_needed), 'constant')
                padded_count += 1
            elif current_length_samples > target_length_samples:
                output_audio = y[:target_length_samples]
                truncated_count += 1
            else:
                already_correct_duration_count += 1

            if operation_mode == 1: 
                output_path_for_this_file = wav_file_path
            else: 
                relative_path = wav_file_path.relative_to(audio_dir)
                output_path_for_this_file = output_dir / relative_path
                output_path_for_this_file.parent.mkdir(parents=True, exist_ok=True)

            if (current_length_samples != target_length_samples) or \
               (operation_mode == 2 and wav_file_path != output_path_for_this_file):
                sf.write(output_path_for_this_file, output_audio, sample_rate)
            elif operation_mode == 2 and wav_file_path == output_path_for_this_file and not output_path_for_this_file.exists():
                 shutil.copy2(wav_file_path, output_path_for_this_file)

        except Exception as e:
            print(f"  ! Error processing file {wav_file_path.name} for duration adjustment: {e}")
            failed_processing_count += 1

    print("Duration adjustment process complete.")
    return total_files, padded_count, truncated_count, already_correct_duration_count, failed_processing_count

if __name__ == "__main__":
    print(f"Input Uniform SR Audio Directory: {UNIFORM_SR_AUDIO_DIRECTORY}")
    print(f"Target Duration: {TARGET_DURATION_SECONDS} seconds")
    print(f"Expected Sample Rate: {EXPECTED_SAMPLE_RATE} Hz")
    print(f"Operation Mode: {OPERATION_MODE} (1=In-place, 2=New Output Dir)")
    if OPERATION_MODE == 2:
        print(f"Target Fixed-Duration Output Directory: {FINAL_FIXED_DURATION_OUTPUT_PATH}\n")
    else:
        print("Files will be modified in-place if their duration differs.\n")

    total, padded, truncated, already_duration, failed = adjust_audio_duration(
        UNIFORM_SR_AUDIO_DIRECTORY,
        TARGET_DURATION_SECONDS,
        EXPECTED_SAMPLE_RATE,
        OPERATION_MODE,
        FINAL_FIXED_DURATION_OUTPUT_PATH if OPERATION_MODE == 2 else None
    )

    print("\n--- Duration Adjustment Summary ---")
    print(f"Total WAV files processed: {total}")
    print(f"Files padded to {TARGET_DURATION_SECONDS}s: {padded}")
    print(f"Files truncated to {TARGET_DURATION_SECONDS}s: {truncated}")
    print(f"Files already at {TARGET_DURATION_SECONDS}s (and copied if mode 2): {already_duration}")
    print(f"Files failed to process: {failed}")

    if failed > 0:
        print("Review logs or uncomment debug print statements for details on failures.")

    if OPERATION_MODE == 1:
        print(f"Files in '{UNIFORM_SR_AUDIO_DIRECTORY}' have had their durations adjusted in-place.")
    elif OPERATION_MODE == 2 and total > 0:
        print(f"All processed files (fixed duration, mono, {EXPECTED_SAMPLE_RATE//1000}kHz SR) are in: {FINAL_FIXED_DURATION_OUTPUT_PATH}")

In [ ]:
# 9- Data Augmentation (Final Corrected Version)
import os
import random
import shutil
from pathlib import Path
from collections import Counter
import numpy as np
import librosa
import soundfile as sf

# --- CONFIGURATION ---
# Ensure this matches your folder from the previous step!
INPUT_AUDIO_DIRECTORY = Path("/kaggle/working/Processed_WAV_Audio/Baby Cry Sence Dataset") 
AUGMENTED_OUTPUT_DIRECTORY_BASE = Path("/kaggle/working/")
AUGMENTED_OUTPUT_FOLDER_NAME = "Balanced_Augmented_Dataset_100_Simple"
FINAL_AUGMENTED_OUTPUT_PATH = AUGMENTED_OUTPUT_DIRECTORY_BASE / AUGMENTED_OUTPUT_FOLDER_NAME

TARGET_FILES_PER_CATEGORY = 100
EXPECTED_SAMPLE_RATE = 16000
TARGET_DURATION_SECONDS = 7.0 
TARGET_DURATION_SAMPLES = int(TARGET_DURATION_SECONDS * EXPECTED_SAMPLE_RATE)

# --- AUGMENTATION FUNCTIONS ---
def augment_shift_time(y, sr):
    shift_max_samples = int(0.05 * len(y)) 
    shift_samples = random.randint(-shift_max_samples, shift_max_samples)
    return np.roll(y, shift_samples)

def augment_change_pitch(y, sr):
    n_steps = random.uniform(-0.6, 0.6) 
    if abs(n_steps) < 0.1: n_steps = 0.2
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def augment_add_noise(y, sr): 
    noise_amplitude = random.uniform(0.0005, 0.0015) * np.max(np.abs(y))
    noise = np.random.normal(0, noise_amplitude, len(y))
    return np.clip(y + noise, -1.0, 1.0) 

augmentation_choices = [
    {"name": "shifted", "func": augment_shift_time},
    {"name": "pitched", "func": augment_change_pitch},
    {"name": "noised", "func": augment_add_noise},
]

def _ensure_fixed_length(y_processed, target_samples):
    current_len = len(y_processed)
    if current_len < target_samples:
        return np.pad(y_processed, (0, target_samples - current_len), 'constant')
    return y_processed[:target_samples]

# --- MAIN AUGMENTATION ENGINE ---
def balance_and_augment_simplified(input_dir: Path, output_dir: Path, target_count: int, sr: int):
    if not input_dir.exists():
        print(f"❌ ERROR: Input directory {input_dir} not found!")
        return

    # Automatically find folders that contain wav files
    all_categories = [d for d in input_dir.iterdir() if d.is_dir() and list(d.glob('*.wav'))]
    
    if not all_categories:
        print(f"❌ ERROR: No .wav files found in subfolders of {input_dir}")
        print(f"Directory contains: {os.listdir(input_dir)}")
        return

    print(f"✅ Found {len(all_categories)} categories. Target: {target_count} files each.\n")

    for category_dir in all_categories:
        category_name = category_dir.name
        cat_out_dir = output_dir / category_name
        cat_out_dir.mkdir(parents=True, exist_ok=True)

        original_files = sorted(list(category_dir.glob('*.wav')))
        num_originals = len(original_files)
        
        print(f"🚀 Processing '{category_name}' ({num_originals} originals)...")

        # 1. Standardize and Copy Originals
        for f_path in original_files:
            try:
                y, _ = librosa.load(f_path, sr=sr)
                y_fixed = _ensure_fixed_length(y, TARGET_DURATION_SAMPLES)
                sf.write(cat_out_dir / f_path.name, y_fixed, sr)
            except Exception as e:
                print(f"  ! Error copying {f_path.name}: {e}")

        # 2. Augment to reach Target
        current_count = len(list(cat_out_dir.glob('*.wav')))
        needed = target_count - current_count

        if needed > 0:
            from itertools import cycle
            file_cycle = cycle(original_files)
            for i in range(needed):
                try:
                    orig_path = next(file_cycle)
                    y_orig, _ = librosa.load(orig_path, sr=sr)
                    
                    choice = random.choice(augmentation_choices)
                    y_aug = choice["func"](y_orig, sr)
                    y_aug = _ensure_fixed_length(y_aug, TARGET_DURATION_SAMPLES)

                    out_name = f"{orig_path.stem}_aug_{choice['name']}_{i}.wav"
                    sf.write(cat_out_dir / out_name, y_aug, sr)
                except Exception as e:
                    pass # Ignore occasional read errors
            
        print(f"  Done. Final count for '{category_name}': {len(list(cat_out_dir.glob('*.wav')))}")

    print("\n🎉 ALL CATEGORIES BALANCED AND AUGMENTED!")

# --- EXECUTION ---
balance_and_augment_simplified(
    INPUT_AUDIO_DIRECTORY, 
    FINAL_AUGMENTED_OUTPUT_PATH, 
    TARGET_FILES_PER_CATEGORY, 
    EXPECTED_SAMPLE_RATE
)

In [ ]:
import shutil
import random
from pathlib import Path

# --- PATHS ---
FINAL_AUGMENTED_OUTPUT_PATH = Path("/kaggle/working/Balanced_Augmented_Dataset_100_Simple")
BALANCED_SUBSET_DIR = Path("/kaggle/working/Balanced_100_Per_Category")

# --- FUNCTION TO CREATE BALANCED SUBSET ---
def create_balanced_subset(source_dir: Path, subset_dir: Path, subset_count: int = 100):
    subset_dir.mkdir(parents=True, exist_ok=True)
    
    categories = [d for d in source_dir.iterdir() if d.is_dir()]
    for category_dir in categories:
        category_name = category_dir.name
        cat_subset_dir = subset_dir / category_name
        cat_subset_dir.mkdir(parents=True, exist_ok=True)
        
        all_files = list(category_dir.glob('*.wav'))
        if len(all_files) <= subset_count:
            selected_files = all_files
        else:
            selected_files = random.sample(all_files, subset_count)  # pick exactly subset_count
        
        for f in selected_files:
            shutil.copy(f, cat_subset_dir / f.name)
        
        print(f"✅ Category '{category_name}': {len(selected_files)} files copied to subset.")

# --- EXECUTION ---
create_balanced_subset(FINAL_AUGMENTED_OUTPUT_PATH, BALANCED_SUBSET_DIR, subset_count=100)


In [ ]:
# 12- MFCC Features
import os
from pathlib import Path
import librosa
import numpy as np
import pandas as pd

# Constants
N_MFCC = 13
N_FFT = int(0.025 * 16000)
HOP_LENGTH = int(0.010 * 16000) 
N_MELS = 40
EXPECTED_SAMPLE_RATE = 16000
TARGET_FRAMES = 701 # For exactly 7 seconds at 16kHz

def extract_mfcc_features(audio_dir: Path, n_mfcc: int, n_fft: int, hop_length: int, n_mels: int,
                           expected_sr: int = 16000):
    if not audio_dir.exists():
        print(f"Error: Directory '{audio_dir}' does not exist.")
        return [], [], [], []

    all_wav_files = sorted([p for p in audio_dir.rglob('*.wav') if p.is_file()])
    total_files = len(all_wav_files)

    if total_files == 0:
        print(f"No .wav files found in '{audio_dir}'.")
        return [], [], [], []

    print(f"🚀 Extracting MFCCs. Target Shape for each: ({n_mfcc}, {TARGET_FRAMES})")

    features_list, labels_list, filenames_list, error_files = [], [], [], []

    for idx, wav_file_path in enumerate(all_wav_files):
        try:
            # Load and extract
            y, sr = librosa.load(wav_file_path, sr=expected_sr, mono=True)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                                         n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
            
            # --- FORCED CONSISTENCY: Ensure exact (13, 701) shape ---
            if mfccs.shape[1] < TARGET_FRAMES:
                pad_width = TARGET_FRAMES - mfccs.shape[1]
                mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
            else:
                mfccs = mfccs[:, :TARGET_FRAMES]
            
            features_list.append(mfccs)
            labels_list.append(wav_file_path.parent.name)
            filenames_list.append(wav_file_path.name)
            
            if (idx + 1) % 500 == 0 or (idx + 1) == total_files:
                print(f"  Processed {idx + 1}/{total_files}...")

        except Exception as e:
            error_files.append(str(wav_file_path))

    return features_list, labels_list, filenames_list, error_files

if __name__ == "__main__":
    # --- SMART PATH REDIRECTION ---
    print("🔍 Searching for your processed audio files...")
    found_audio_path = None
    for root, dirs, files in os.walk("/kaggle/working/"):
        if any(f.endswith(".wav") for f in files):
            if "Balanced" in root or "100_Per_Category" in root:
                found_audio_path = Path(root)
                break
            found_audio_path = Path(root)

    if found_audio_path:
        FINAL_PROCESSED_AUDIO_DIR = found_audio_path
        print(f"✅ Found audio files in: {FINAL_PROCESSED_AUDIO_DIR}")
        
        # Run Extraction
        mfcc_features, labels, filenames, errors = extract_mfcc_features(
            FINAL_PROCESSED_AUDIO_DIR, N_MFCC, N_FFT, HOP_LENGTH, N_MELS, EXPECTED_SAMPLE_RATE
        )

        if mfcc_features:
            output_feature_dir = Path("/kaggle/working/MFCC_Features/")
            output_feature_dir.mkdir(parents=True, exist_ok=True)
            
            # Since we forced consistency in the function, stacking will always work now
            features_array = np.stack(mfcc_features, axis=0)
            np.save(output_feature_dir / "mfcc_features_stacked.npy", features_array)
            np.save(output_feature_dir / "labels_stacked.npy", np.array(labels))
            
            print(f"\n✅ SUCCESS! All features are unified.")
            print(f"Stacked array shape: {features_array.shape} (Samples, Coefficients, Frames)")
            print(f"Saved to: {output_feature_dir}")
    else:
        print("❌ ERROR: Could not find augmented dataset. Please run Step 9.")

In [ ]:
# 13- Mel-Spectrogram Features
import os
from pathlib import Path
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

# --- SMART PATH REDIRECTION ---
print("🔍 Searching for your processed audio files...")
found_audio_path = None
for root, dirs, files in os.walk("/kaggle/working/"):
    if any(f.endswith(".wav") for f in files):
        if "Balanced" in root or "100_Per_Category" in root:
            found_audio_path = Path(root)
            break
        found_audio_path = Path(root)

if found_audio_path:
    FINAL_PROCESSED_AUDIO_DIR = found_audio_path
    print(f"✅ Found audio files in: {FINAL_PROCESSED_AUDIO_DIR}")
else:
    raise FileNotFoundError("❌ ERROR: No .wav files found in /kaggle/working/. Run Step 9 first.")

# Constants
EXPECTED_SAMPLE_RATE = 16000
N_FFT = int(0.025 * EXPECTED_SAMPLE_RATE)
HOP_LENGTH = int(0.010 * EXPECTED_SAMPLE_RATE)
N_MELS = 128
FMIN = 0
FMAX = EXPECTED_SAMPLE_RATE / 2
OUTPUT_FEATURE_DIR = Path("/kaggle/working/Mel_Spectrogram_Features/")
TARGET_FRAMES = 701 # For exactly 7 seconds at 16kHz

def extract_mel_spectrogram_features(audio_dir: Path, n_fft, hop_length, n_mels, fmin, fmax):
    all_wav_files = sorted([p for p in audio_dir.rglob('*.wav') if p.is_file()])
    total_files = len(all_wav_files)
    
    features_list, labels_list, filenames_list = [], [], []

    print(f"🚀 Processing {total_files} files...")

    for idx, wav_path in enumerate(all_wav_files):
        try:
            y, sr = librosa.load(wav_path, sr=EXPECTED_SAMPLE_RATE, mono=True)
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, 
                                               hop_length=hop_length, n_mels=n_mels, 
                                               fmin=fmin, fmax=fmax)
            S_db = librosa.power_to_db(S, ref=np.max)

            # --- FORCED RESIZING FOR CONSISTENCY ---
            if S_db.shape[1] < TARGET_FRAMES:
                pad_width = TARGET_FRAMES - S_db.shape[1]
                S_db = np.pad(S_db, ((0, 0), (0, pad_width)), mode='constant', constant_values=S_db.min())
            else:
                S_db = S_db[:, :TARGET_FRAMES]

            features_list.append(S_db)
            labels_list.append(wav_path.parent.name)
            filenames_list.append(wav_path.name)
            
            if (idx + 1) % 500 == 0:
                print(f"  Processed {idx + 1}/{total_files}")
        except Exception as e:
            print(f"Skip {wav_path.name}: {e}")

    return np.array(features_list), np.array(labels_list), np.array(filenames_list)

if __name__ == "__main__":
    OUTPUT_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    
    X, y, files = extract_mel_spectrogram_features(FINAL_PROCESSED_AUDIO_DIR, N_FFT, HOP_LENGTH, N_MELS, FMIN, FMAX)

    if len(X) > 0:
        np.save(OUTPUT_FEATURE_DIR / "mel_spectrogram_features_stacked.npy", X)
        np.save(OUTPUT_FEATURE_DIR / "mel_spectrogram_labels_stacked.npy", y)
        print(f"\n✅ SUCCESS! Saved stacked features with shape: {X.shape}")
        
        # Visualize the first one
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(X[0], x_axis='time', y_axis='mel', sr=EXPECTED_SAMPLE_RATE)
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"Mel Spectrogram: {y[0]}")
        plt.show()

In [ ]:
VGGISH_MODEL = "https://tfhub.dev/google/vggish/1"
vggish = hub.load(VGGISH_MODEL)

In [ ]:
import librosa
import numpy as np

def load_wav_16k(file_path):
    wav, sr = librosa.load(file_path, sr=16000, mono=True)
    return wav.astype(np.float32)

In [ ]:
X_embeddings = []
y_labels = []

AUDIO_DIR = "/kaggle/working/Balanced_100_Per_Category"

class_dirs = sorted([d for d in Path(AUDIO_DIR).iterdir() if d.is_dir()])
label_encoder.fit([d.name for d in class_dirs])

for class_dir in class_dirs:
    label = label_encoder.transform([class_dir.name])[0]

    for wav_file in class_dir.glob("*.wav"):
        waveform = load_wav_16k(str(wav_file))

        # ✅ correct input
        embeddings = vggish(waveform)   # (num_frames, 128)

        # temporal pooling
        embedding = embeddings.numpy().mean(axis=0)

        X_embeddings.append(embedding)
        y_labels.append(label)

X = np.array(X_embeddings)
y = np.array(y_labels)

print(X.shape)  # (N, 128)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4),
          input_shape=(128,)),
    Dropout(0.5),
    Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping
CSV_LOG_PATH = Path('/kaggle/working/vggish_cnn.csv')
csv_logger = CSVLogger(CSV_LOG_PATH, append=False)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[csv_logger, early_stop],
    verbose=1
)

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

YAMNET_MODEL_HANDLE = "https://tfhub.dev/google/yamnet/1"

yamnet_model = hub.load(YAMNET_MODEL_HANDLE)
print("YAMNet loaded successfully")


In [ ]:
import librosa
import numpy as np

TARGET_SR = 16000
TARGET_DURATION = 7.0

def load_wav_16k_mono(file_path):
    wav, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)
    expected_len = int(TARGET_SR * TARGET_DURATION)

    if len(wav) < expected_len:
        wav = np.pad(wav, (0, expected_len - len(wav)))
    else:
        wav = wav[:expected_len]

    return wav.astype(np.float32)


In [ ]:
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

X_embeddings = []
y_labels = []
AUDIO_DIR = "/kaggle/working/Balanced_100_Per_Category"
class_folders = sorted([d for d in Path(AUDIO_DIR).iterdir() if d.is_dir()])

label_encoder = LabelEncoder()
class_names = [d.name for d in class_folders]
label_encoder.fit(class_names)

for class_dir in class_folders:
    label = class_dir.name
    encoded_label = label_encoder.transform([label])[0]

    for wav_file in class_dir.glob("*.wav"):
        waveform = load_wav_16k_mono(str(wav_file))
        scores, embeddings, spectrogram = yamnet_model(waveform)
        embedding = tf.reduce_mean(embeddings, axis=0).numpy()

        X_embeddings.append(embedding)
        y_labels.append(encoded_label)

X = np.array(X_embeddings)
y = np.array(y_labels)

print("Final dataset shape:", X.shape, y.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import CSVLogger

NUM_CLASSES = len(label_encoder.classes_)

model = Sequential([
    Dense(256, activation='relu', input_shape=(1024,)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
EPOCHS = 20
BATCH_SIZE = 32
CSV_LOG_PATH = Path('/kaggle/working/yamnet_cnn.csv')
csv_logger = CSVLogger(CSV_LOG_PATH, append=False)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[csv_logger],
    verbose=1
)


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Train Accuracy: {train_acc:.4f}, Train Loss: {train_loss:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}, Test Loss:  {test_loss:.4f}")


In [ ]:
import pickle
MODEL_SAVE_PATH = Path('/kaggle/working/yamnet_cnn_model1.pkl')
LABEL_ENCODER_PATH = Path('/kaggle/working/yamnet_cnn_model_label1.pkl')
with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump(model, f)

with open(LABEL_ENCODER_PATH, "wb") as f:
    pickle.dump(label_encoder, f)

print("Model and Label Encoder saved successfully")


In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Loss plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

plt.figure(figsize=(7, 7))
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix – Baby Cry Classification')
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)


In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Loss plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()
